In [ ]:
# Cell 1 — FTMO-confirmed crypto daily panel from Binance hourly pool
import pandas as pd
from backtest.data import DataPanel

P = r"C:\Users\User\backtest_engine\backtest_engine2\data\binance_hourly_top50_pool.parquet"
df = pd.read_parquet(P)

# FTMO-confirmed tradables (intersection with pool); stables/pegs excluded by construction
FTMO14 = ["BTC","ETH","ADA","DOT","DASH","LTC","XRP","DOGE","SOL","BNB","XLM","AAVE","LINK","AVAX"]
KEEP = {c + "USDT" for c in FTMO14}
df = df[df["symbol"].isin(KEEP)].copy()
df["coin_vol"] = df["quote_volume"] / df["close"]          # quote($) -> coin count for ADV

close = df.pivot_table(index="open_time", columns="symbol", values="close")
cvol  = df.pivot_table(index="open_time", columns="symbol", values="coin_vol")
close.index = pd.to_datetime(close.index).tz_localize(None)
cvol.index  = pd.to_datetime(cvol.index).tz_localize(None)

prices_skew = close.resample("1D").last().sort_index()
volume_skew = cvol.resample("1D").sum().reindex(index=prices_skew.index, columns=prices_skew.columns)
ren = {c: c[:-4] for c in prices_skew.columns}
prices_skew = prices_skew.rename(columns=ren); volume_skew = volume_skew.rename(columns=ren)
prices_skew.index.name = volume_skew.index.name = "Date"

# truncate ragged tail: 11/14 coins' data collection ends 2026-05-19
CUTOFF = "2026-05-19"
prices_skew = prices_skew.loc[:CUTOFF]
volume_skew = volume_skew.loc[:CUTOFF]

missing = sorted(set(FTMO14) - set(prices_skew.columns))
panel_skew = DataPanel(prices_skew, volume=volume_skew, check_outliers=True)
print(f"Bars:   {len(panel_skew.dates)}")
print(f"Assets: {len(panel_skew.assets_all)}  (missing from pool: {missing or 'none'})")
print(f"Range:  {panel_skew.dates[0].date()} -> {panel_skew.dates[-1].date()}")
print(f"NaN fraction: {prices_skew.isna().mean().mean():.1%}")
live = prices_skew.notna().sum(axis=1)
print(f"Coins live: {live.iloc[0]} at start -> {live.iloc[-1]} at end")
dvol = (prices_skew * volume_skew).median()
q = dvol.quantile([0.05, 0.5, 0.95]) / 1e6
print(f"Median daily $-vol per coin ($M): p5={q.iloc[0]:.1f} median={q.iloc[1]:.1f} p95={q.iloc[2]:.0f}")


In [ ]:
# Cell 2 — SkewLottery strategy: short lottery-shaped (high 21d skew), long steady
import pandas as pd
from backtest.strategy import Strategy
from backtest.risk import RiskConfig

class SkewLottery(Strategy):
    """Cross-sectional lottery-asymmetry alpha. Signal = -rank(skew of daily returns,
    trailing `window` days), demeaned, linear-decayed over `decay` days, gross 1.0.
    Hypothesis: skew-seeking buyers overpay for lottery-like payoffs -> high skew underperforms."""
    rebalance_frequency = "daily"
    risk = RiskConfig(max_position=0.15, max_gross=1.0, max_net=1.0)

    def __init__(self, window: int = 21, decay: int = 5):
        self.window = window
        self.decay = decay

    def __repr__(self):
        return f"SkewLottery(window={self.window}, decay={self.decay})"

    def generate_weights(self, data, t):
        w, d = self.window, self.decay
        px = data.prices
        if len(px) < w + d + 1:
            return pd.Series(0.0, index=data.assets)
        dr = px[data.assets].pct_change(fill_method=None).iloc[-(w + d - 1):]
        sk = dr.rolling(w, min_periods=w).skew()                # full window only
        num = pd.Series(0.0, index=dr.columns)
        den = pd.Series(0.0, index=dr.columns)
        for k in range(d):                                      # k=0 newest, weight d-k
            row = sk.iloc[-(k + 1)].dropna()
            if len(row) < 5:
                continue
            r = row.rank()
            s = -(r - r.mean())                                 # short high skew, long low
            wt = float(d - k)
            num = num.add(s * wt, fill_value=0.0)
            den = den.add(pd.Series(wt, index=s.index), fill_value=0.0)
        valid = den > 0
        if valid.sum() < 5:
            return pd.Series(0.0, index=data.assets)
        sig = num[valid] / den[valid]
        sig = sig - sig.mean()                                  # re-demean after decay
        gross = sig.abs().sum()
        if gross == 0:
            return pd.Series(0.0, index=data.assets)
        return sig / gross

strat_skew = SkewLottery(window=21, decay=5)
print(strat_skew)
rb = strat_skew.rebalance_dates(panel_skew.dates)
yrs = (panel_skew.dates[-1] - panel_skew.dates[0]).days / 365.25
print(f"Rebalances/year: {len(rb) / yrs:.0f}")
v = panel_skew.as_of(panel_skew.dates[-1])
wts = strat_skew.generate_weights(v, panel_skew.dates[-1])
print(f"Last bar: nonzero={int((wts != 0).sum())}  gross={wts.abs().sum():.3f}  "
      f"net={wts.sum():+.3e}  max|w|={wts.abs().max():.2%}")


In [ ]:
# Cell 3 — FTMO-calibrated crypto CFD cost stack
from backtest.costs import (
    Commission, Spread, MarketImpact, ShortBorrow,
    CompositeCostModel, LiquidityCap,
)

# Symmetric CFD swap S=5%/yr per side; dollar-neutral book (shorts = gross/2)
# => model both legs exactly as ShortBorrow(2*S) on the short notional.
costs_ftmo = CompositeCostModel([
    Commission(bps=3.25),                               # FTMO published 0.0325%/side
    Spread(half_bps=5),                                 # blended majors/alts (as #005/#006)
    MarketImpact(k=12, kind="sqrt", adv_lookback=20),
    ShortBorrow(annual_bps=1000, trading_days=365),     # 2*S, S=500bp/side baseline
])
liq_ftmo = LiquidityCap(cap_pct=0.05, adv_lookback=20)

print("Cost components:", [type(m).__name__ for m in costs_ftmo.models])
print(f"LiquidityCap: cap_pct={liq_ftmo.cap_pct}, adv_lookback={liq_ftmo.adv_lookback}")
print(f"panel_skew has volume: {panel_skew.has_field('volume')}")


In [ ]:
# Cell 4 — Single-path run: SkewLottery on FTMO-14, FTMO cost stack
from backtest.engine import Engine

WARMUP_BARS = 30   # strategy needs 26 (21 window + 5 decay)

engine_skew = Engine(
    panel_skew,
    strat_skew,
    costs=costs_ftmo,
    liquidity=liq_ftmo,
    initial_capital=1_000_000,
)
result_skew = engine_skew.run(start=panel_skew.dates[WARMUP_BARS])

print(f"Bars:          {len(result_skew.equity_curve)}")
print(f"Rebalances:    {result_skew.metadata['n_rebalances']}")
print(f"Date range:    {result_skew.metadata['start'].date()} -> {result_skew.metadata['end'].date()}")
print(f"Initial:       ${result_skew.metadata['initial_capital']:>14,.0f}")
print(f"Final equity:  ${result_skew.equity_curve.iloc[-1]:>14,.0f}")
print(f"Total return:  {(result_skew.equity_curve.iloc[-1] / result_skew.metadata['initial_capital'] - 1):+.2%}")
print(f"Total costs:   ${result_skew.costs.sum():>14,.0f}")
print(f"Max position weight:  {result_skew.weights.abs().max().max():.2%}")
print(f"Max gross exposure:   {result_skew.weights.abs().sum(axis=1).max():.2%}")


In [ ]:
# Cell 5 — Metrics report for the short-lottery run
from backtest.metrics import compute_metrics, sharpe_ratio

rep_skew = compute_metrics(result_skew.returns, result_skew.equity_curve,
                           costs=result_skew.costs, trades=result_skew.trades,
                           ann_factor=365)
yrs_s = (result_skew.equity_curve.index[-1] - result_skew.equity_curve.index[0]).days / 365.25
gS_s = sharpe_ratio(result_skew.gross_pnl / result_skew.equity_curve.shift(1), ann_factor=365)

print(rep_skew)
print()
print(f"NET   ret:    {result_skew.equity_curve.iloc[-1]/1e6-1:+.2%}   over {yrs_s:.1f}y")
print(f"NET   Sharpe: {rep_skew.sharpe:+.3f}  (95% CI [{rep_skew.sharpe_ci_low:+.2f}, {rep_skew.sharpe_ci_high:+.2f}])")
print(f"GROSS Sharpe: {gS_s:+.3f}")
print(f"Turnover:     {rep_skew.turnover/yrs_s:.0f}x/yr   Margin: {rep_skew.margin*1e4:+.2f} bps/$")
print(f"Max DD:       {rep_skew.max_drawdown:.1%}   Ann vol: {rep_skew.ann_vol:.1%}")
print(f"PSR(0):       {rep_skew.psr:.3f}")


In [ ]:
# Cell 6 — Trial 2: flipped sign = LONG lottery-skew, short steady (user's original hypothesis)
class SkewLotteryFlipped(SkewLottery):
    """Long high trailing skew (lottery-shaped), short low. Underreaction-to-surges story."""
    def __repr__(self):
        return f"SkewLotteryFlipped(window={self.window}, decay={self.decay})"
    def generate_weights(self, data, t):
        return -super().generate_weights(data, t)

strat_flip = SkewLotteryFlipped(window=21, decay=5)
result_flip = Engine(panel_skew, strat_flip, costs=costs_ftmo, liquidity=liq_ftmo,
                     initial_capital=1_000_000).run(start=panel_skew.dates[WARMUP_BARS])

rep_flip = compute_metrics(result_flip.returns, result_flip.equity_curve,
                           costs=result_flip.costs, trades=result_flip.trades,
                           ann_factor=365)
yrs_f = (result_flip.equity_curve.index[-1] - result_flip.equity_curve.index[0]).days / 365.25
gS_f = sharpe_ratio(result_flip.gross_pnl / result_flip.equity_curve.shift(1), ann_factor=365)

print(strat_flip)
print(f"NET   ret:    {result_flip.equity_curve.iloc[-1]/1e6-1:+.2%}   over {yrs_f:.1f}y")
print(f"NET   Sharpe: {rep_flip.sharpe:+.3f}  (95% CI [{rep_flip.sharpe_ci_low:+.2f}, {rep_flip.sharpe_ci_high:+.2f}])")
print(f"GROSS Sharpe: {gS_f:+.3f}")
print(f"Turnover:     {rep_flip.turnover/yrs_f:.0f}x/yr   Margin: {rep_flip.margin*1e4:+.2f} bps/$")
print(f"Max DD:       {rep_flip.max_drawdown:.1%}   Ann vol: {rep_flip.ann_vol:.1%}")
print(f"PSR(0):       {rep_flip.psr:.3f}")


In [ ]:
# Cell 7 — #006 replica (30d XS momentum) on FTMO-14, correlate vs flipped skew (Step 6)
import pandas as pd

class Momentum30(SkewLottery):
    """#006's final construction: +rank(clipped 30d return), demeaned, decay 5, gross 1.0."""
    def __init__(self, lookback: int = 30, clip: float = 1.0, decay: int = 5):
        self.lookback = lookback
        self.clip = clip
        self.decay = decay

    def __repr__(self):
        return f"Momentum30(lookback={self.lookback}, clip={self.clip}, decay={self.decay})"

    def generate_weights(self, data, t):
        k, d = self.lookback, self.decay
        px = data.prices
        if len(px) < k + d + 1:
            return pd.Series(0.0, index=data.assets)
        dr = px[data.assets].pct_change(fill_method=None).clip(-self.clip, self.clip)
        num = pd.Series(0.0, index=dr.columns)
        den = pd.Series(0.0, index=dr.columns)
        for j in range(d):                                   # j=0 newest, weight d-j
            win = dr.iloc[len(dr) - k - j: len(dr) - j]
            valid = win.notna().all()
            ret = ((1.0 + win).prod() - 1.0)[valid]
            if len(ret) < 5:
                continue
            r = ret.rank()
            s = r - r.mean()                                 # momentum: winners long
            wt = float(d - j)
            num = num.add(s * wt, fill_value=0.0)
            den = den.add(pd.Series(wt, index=s.index), fill_value=0.0)
        ok = den > 0
        if ok.sum() < 5:
            return pd.Series(0.0, index=data.assets)
        sig = num[ok] / den[ok]
        sig = sig - sig.mean()
        gross = sig.abs().sum()
        if gross == 0:
            return pd.Series(0.0, index=data.assets)
        return sig / gross

strat_m30 = Momentum30()
result_m30 = Engine(panel_skew, strat_m30, costs=costs_ftmo, liquidity=liq_ftmo,
                    initial_capital=1_000_000).run(start=panel_skew.dates[40])  # 30+5 warmup

rep_m30 = compute_metrics(result_m30.returns, result_m30.equity_curve,
                          costs=result_m30.costs, trades=result_m30.trades, ann_factor=365)
yrs_m30 = (result_m30.equity_curve.index[-1] - result_m30.equity_curve.index[0]).days / 365.25
gS_m30 = sharpe_ratio(result_m30.gross_pnl / result_m30.equity_curve.shift(1), ann_factor=365)
print(strat_m30)
print(f"#006-on-FTMO14: GROSS Sh {gS_m30:+.3f} | NET Sh {rep_m30.sharpe:+.3f} | "
      f"turnover {rep_m30.turnover/yrs_m30:.0f}x/yr | maxDD {rep_m30.max_drawdown:.0%}")
print()
both = pd.concat([result_flip.returns.rename("skew_flip"),
                  result_m30.returns.rename("mom30")], axis=1).dropna()
corr = both["skew_flip"].corr(both["mom30"])
print(f"Overlap bars: {len(both)}")
print(f"Pearson corr(flip skew, mom30): {corr:+.3f}")
print("Verdict:", "REDUNDANT (>0.7)" if corr > 0.7 else
      "ok (0.3-0.5)" if corr > 0.3 else "GOOD diversifier (<0.3)" if corr < 0.3 else "borderline")


In [ ]:
# Cell 8 — Trial 3: flipped skew with window=30 (horizon aligned to #006's 30d plateau)
strat_flip30 = SkewLotteryFlipped(window=30, decay=5)
result_flip30 = Engine(panel_skew, strat_flip30, costs=costs_ftmo, liquidity=liq_ftmo,
                       initial_capital=1_000_000).run(start=panel_skew.dates[40])

rep_f30 = compute_metrics(result_flip30.returns, result_flip30.equity_curve,
                          costs=result_flip30.costs, trades=result_flip30.trades,
                          ann_factor=365)
yrs_f30 = (result_flip30.equity_curve.index[-1] - result_flip30.equity_curve.index[0]).days / 365.25
gS_f30 = sharpe_ratio(result_flip30.gross_pnl / result_flip30.equity_curve.shift(1), ann_factor=365)

print(strat_flip30)
print(f"NET   ret:    {result_flip30.equity_curve.iloc[-1]/1e6-1:+.2%}   over {yrs_f30:.1f}y")
print(f"NET   Sharpe: {rep_f30.sharpe:+.3f}  (95% CI [{rep_f30.sharpe_ci_low:+.2f}, {rep_f30.sharpe_ci_high:+.2f}])")
print(f"GROSS Sharpe: {gS_f30:+.3f}")
print(f"Turnover:     {rep_f30.turnover/yrs_f30:.0f}x/yr   Margin: {rep_f30.margin*1e4:+.2f} bps/$")
print(f"Max DD:       {rep_f30.max_drawdown:.1%}   Ann vol: {rep_f30.ann_vol:.1%}")
print(f"PSR(0):       {rep_f30.psr:.3f}")

both30 = pd.concat([result_flip30.returns.rename("skew30"),
                    result_m30.returns.rename("mom30")], axis=1).dropna()
print(f"corr(skew30, mom30): {both30['skew30'].corr(both30['mom30']):+.3f}")


In [ ]:
# Cell 9 — Swap sensitivity for flip30: S = 0 / 5 / 15 %/yr per side (cost scenarios, not new trials)
for label, sbps in [("swap-free (0%/side)", 0), ("baseline (5%/side)", 1000), ("pessimistic (15%/side)", 3000)]:
    comps = [Commission(bps=3.25), Spread(half_bps=5), MarketImpact(k=12, kind="sqrt", adv_lookback=20)]
    if sbps > 0:
        comps.append(ShortBorrow(annual_bps=sbps, trading_days=365))   # 2*S on shorts = S on both legs
    c = CompositeCostModel(comps)
    res = Engine(panel_skew, SkewLotteryFlipped(window=30, decay=5), costs=c, liquidity=liq_ftmo,
                 initial_capital=1_000_000).run(start=panel_skew.dates[40])
    rep = compute_metrics(res.returns, res.equity_curve, costs=res.costs, trades=res.trades,
                          ann_factor=365)
    yrs = (res.equity_curve.index[-1] - res.equity_curve.index[0]).days / 365.25
    print(f"{label:24s} NET Sh {rep.sharpe:+.3f}  net ret {res.equity_curve.iloc[-1]/1e6-1:+8.1%}  "
          f"margin {rep.margin*1e4:+6.2f} bps/$  maxDD {rep.max_drawdown:.0%}  PSR {rep.psr:.3f}")


In [ ]:
# Cell 10 — DSR for flip30 (baseline costs), deflated by this session's trials
import numpy as np
from backtest.selection import dsr, effective_k, expected_max_sr
from backtest.metrics.core import sharpe_var_term

trials = pd.concat([
    result_skew.returns.rename("short21"),
    result_flip.returns.rename("flip21"),
    result_flip30.returns.rename("flip30"),
    result_m30.returns.rename("mom30"),
], axis=1).dropna()

K_raw = 4
K_eff = effective_k(trials, threshold=0.5)

r = result_flip30.returns.dropna().to_numpy()
T = len(r)
sr_pb = r.mean() / r.std(ddof=1)
var_term = sharpe_var_term(r, sr_pb)
print("Trial corr matrix:")
print(trials.corr().round(2).to_string())
print()
print(f"K raw = {K_raw} | K effective (thr 0.5) = {K_eff}")
for K in sorted({K_raw, K_eff, 1}):
    luck_ann = expected_max_sr(K, var_term / T) * np.sqrt(365)
    print(f"  K={K}: luck-max {luck_ann:.3f} ann | DSR {dsr(result_flip30.returns, K=K):.3f}")
print(f"Realized Sharpe (ann.): {sr_pb * np.sqrt(365):+.3f}")
